# 03 · Build the modeling table

Combines the cleaned competition tables (`01_preprocessing`) with the Census ZIP features (`02_external_census`) into **one row per policy**, ready for modeling.

| Step | What happens |
|---|---|
| 1. Driver roll-up | ~2.2 drivers per policy → age, risk, demographic summaries (`DRV_*`) |
| 2. Vehicle roll-up | ~3.4 vehicles per policy → age, mileage, ownership, safety, make summaries (`VEH_*`) |
| 3. Census join | ACS features joined on `ZIP_CD`, with gaps filled from population-weighted `ZIP3` averages (`ACS_*`) |
| 4. Date parts | `QUOTE_DT` → year / month / day of week |
| 5. Checks & save | One row per policy, no rows lost → `data/processed/model_table.parquet` |

**Deliberately not done here:**
- Anything that uses the target, such as target encoding of `ZIP_CD`, `ZIP3` or `PRIOR_CARRIER_GRP`. It must be computed inside CV folds in the modeling notebook to avoid leakage.
- Engineered price-competitiveness and interaction features. Those come next, once a baseline exists to measure them against.

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

P = '../data/processed'
pol, drv, veh = (pd.read_parquet(f'{P}/{n}.parquet') for n in ('policies', 'drivers', 'vehicles'))
acs = pd.read_parquet('../data/external/acs2018_zcta_features.parquet')

## 1. Driver roll-up
Drivers in a policy have no order (no "primary driver" column), so only order-free summaries are used.
- **Age:** min / mean / max, plus shares of young (<25) and senior (65+) drivers. The youngest driver drives most auto pricing.
- **Risk:** min / mean `SAFETY_RATING` and `ACCIDENT_FREE_YRS`. The minimum is the riskiest driver.
- **Demographics:** shares of male, married, homeowner (`own`), renter, dependent and high-education drivers. `HIGH_EDU_IND` NaN is skipped by the mean.
- **Behavior:** whether any driver opted into telematics, the share who did, and total / max daily commute miles.

The driver count isn't repeated, since `NUM_DRIVERS` already equals the driver row count (checked in `01`).

In [2]:
d = drv.assign(YOUNG=drv.DRIVER_AGE < 25, SENIOR=drv.DRIVER_AGE >= 65, MALE=drv.GENDER == 'M', MARRIED=drv.MARITAL_STATUS == 'married',
               OWN=drv.LIVING_STATUS == 'own', RENT=drv.LIVING_STATUS == 'rent', DEPENDENT=drv.LIVING_STATUS == 'dependent',
               HIGH_EDU=drv.HIGH_EDU_IND.astype('float'))
drv_agg = d.groupby('POLICY_ID').agg(
    DRV_AGE_MIN=('DRIVER_AGE', 'min'), DRV_AGE_MEAN=('DRIVER_AGE', 'mean'), DRV_AGE_MAX=('DRIVER_AGE', 'max'),
    DRV_YOUNG_SHARE=('YOUNG', 'mean'), DRV_SENIOR_SHARE=('SENIOR', 'mean'),
    DRV_SAFETY_MIN=('SAFETY_RATING', 'min'), DRV_SAFETY_MEAN=('SAFETY_RATING', 'mean'),
    DRV_ACC_FREE_MIN=('ACCIDENT_FREE_YRS', 'min'), DRV_ACC_FREE_MEAN=('ACCIDENT_FREE_YRS', 'mean'),
    DRV_MALE_SHARE=('MALE', 'mean'), DRV_MARRIED_SHARE=('MARRIED', 'mean'), DRV_OWN_SHARE=('OWN', 'mean'),
    DRV_RENT_SHARE=('RENT', 'mean'), DRV_DEPENDENT_SHARE=('DEPENDENT', 'mean'), DRV_HIGH_EDU_SHARE=('HIGH_EDU', 'mean'),
    DRV_TELEMATICS_ANY=('TELEMATICS_OPT_IN', 'max'), DRV_TELEMATICS_SHARE=('TELEMATICS_OPT_IN', 'mean'),
    DRV_COMMUTE_SUM=('COMMUTE_MILES_DAILY', 'sum'), DRV_COMMUTE_MAX=('COMMUTE_MILES_DAILY', 'max'))

## 2. Vehicle roll-up
- `VEH_N_ROWS` counts rows in the vehicle table. It's kept **alongside** the policy-level `TOTAL_NUM_VEH`, because the two disagree for ~80% of policies (see `01`); modeling will show which carries signal. The ownership counts are built the same way (`VEH_N_OWNED` etc. vs `NUM_OWNED_VEH`).
- **Age, mileage, safety:** min / mean / max where the extremes matter. `VEHICLE_AGE` NaN (391 rows) is skipped.
- **Protection:** shares with anti-theft and garage.
- **Make:** number of distinct makes, and the share of luxury makes (BMW, Mercedes-Benz, Audi, Cadillac, Acura). Luxury vehicles convert at ~12% vs ~10% for domestic makes. `MAKE_MODEL` itself (1,049 levels) is left for target encoding in CV.

In [3]:
LUXURY = ['BMW', 'MERCEDES-BENZ', 'AUDI', 'CADILLAC', 'ACURA']
v = veh.assign(OWNED=veh.OWNERSHIP_TYPE == 'owned', LEASED=veh.OWNERSHIP_TYPE == 'leased', LOANED=veh.OWNERSHIP_TYPE == 'loaned',
               LUXURY=veh.MAKE.isin(LUXURY))
veh_agg = v.groupby('POLICY_ID').agg(
    VEH_N_ROWS=('CAR_NO', 'size'), VEH_N_OWNED=('OWNED', 'sum'), VEH_N_LEASED=('LEASED', 'sum'), VEH_N_LOANED=('LOANED', 'sum'),
    VEH_AGE_MIN=('VEHICLE_AGE', 'min'), VEH_AGE_MEAN=('VEHICLE_AGE', 'mean'), VEH_AGE_MAX=('VEHICLE_AGE', 'max'),
    VEH_MILEAGE_SUM=('ANNUAL_MILEAGE', 'sum'), VEH_MILEAGE_MAX=('ANNUAL_MILEAGE', 'max'),
    VEH_SAFETY_MIN=('SAFETY_FEATURE_SCORE', 'min'), VEH_SAFETY_MEAN=('SAFETY_FEATURE_SCORE', 'mean'),
    VEH_ANTI_THEFT_SHARE=('ANTI_THEFT_IND', 'mean'), VEH_GARAGE_SHARE=('GARAGE_IND', 'mean'),
    VEH_N_MAKES=('MAKE', 'nunique'), VEH_LUXURY_SHARE=('LUXURY', 'mean'))

## 3. Census join
- Joined on `ZIP_CD` → ZCTA. All 1,168 policy ZIPs match (see `02`).
- **Gap fill:** some ZCTAs are too small for the Census to publish medians or shares, which affects ~4% of policies. Those values are filled with the **population-weighted mean of the same `ZIP3`**. `ACS_ZIP3_FILLED` = 1 marks rows where any value was filled.
- **Missing ZIP:** the 352 policies with no `ZIP_CD` (~1%) have no geography, so their `ACS_*` values stay NaN.
- `ACS_POPULATION` and `ACS_LAND_SQMI` describe ZIP size and are only meaningful for the exact ZIP. They're never missing, so they're never filled.

In [4]:
acs_cols = acs.columns.tolist()
w, zip3 = acs.ACS_POPULATION, acs.index.str[:3]
zip3_mean = (acs.mul(w, axis=0).groupby(zip3).sum(min_count=1) / acs.notna().mul(w, axis=0).groupby(zip3).sum()).replace([np.inf, -np.inf], np.nan)

geo = pol[['POLICY_ID', 'ZIP_CD', 'ZIP3']].join(acs, on='ZIP_CD')
filled = geo[acs_cols].fillna(zip3_mean.reindex(geo.ZIP3).set_axis(geo.index))
geo = geo[['POLICY_ID']].join(filled).assign(ACS_ZIP3_FILLED=(geo[acs_cols].isna() & filled.notna()).any(axis=1).astype('int8'))
print(f"filled from ZIP3: {geo.ACS_ZIP3_FILLED.mean():.1%} of policies; still missing income: {geo.ACS_MED_HH_INCOME.isna().mean():.1%} (= no ZIP)")

filled from ZIP3: 3.8% of policies; still missing income: 1.0% (= no ZIP)


## 4. Merge + date parts
- All tables are left-joined onto policies (`validate='1:1'` guards against duplicating rows).
- `QUOTE_DT` is split into `QUOTE_YEAR`, `QUOTE_MONTH` and `QUOTE_DOW` (0 = Monday) for seasonality. The raw date is kept for reference but **is not a model input**.

In [5]:
df = (pol.merge(drv_agg, on='POLICY_ID', validate='1:1').merge(veh_agg, on='POLICY_ID', validate='1:1')
         .merge(geo, on='POLICY_ID', validate='1:1')
         .assign(QUOTE_YEAR=lambda x: x.QUOTE_DT.dt.year, QUOTE_MONTH=lambda x: x.QUOTE_DT.dt.month, QUOTE_DOW=lambda x: x.QUOTE_DT.dt.day_of_week))

## 5. Checks & save
- One row per policy, all 36,871 present, holdout target still empty, no duplicate columns.
- **Single-feature AUC on the labeled rows** (train + valid) is a quick sanity check that the joins carry signal. Each numeric column is scored alone; values below 0.5 are flipped, because only strength matters here. This is not a measure of model performance.

**Columns for modeling:**
- **IDs / labels (not inputs):** `POLICY_ID`, `SPLIT`, `CONVERT_IND`, `QUOTE_DT`.
- **High-cardinality strings, to target-encode in CV or drop:** `ZIP_CD`, `ZIP3`.
- **Inputs:** everything else. Categoricals are already `category` dtype, which LightGBM and CatBoost accept natively.

In [6]:
assert len(df) == len(pol) == df.POLICY_ID.nunique() == 36871 and df.columns.is_unique
assert df.loc[df.SPLIT == 'holdout', 'CONVERT_IND'].isna().all()
df.to_parquet(f'{P}/model_table.parquet', index=False)

lab = df[df.CONVERT_IND.notna()]
num = lab.select_dtypes('number').columns.drop('CONVERT_IND')
auc = pd.Series({c: roc_auc_score(lab.CONVERT_IND, lab[c].fillna(lab[c].median())) for c in num})
print(df.shape, '| new columns by prefix:', {p: df.columns.str.startswith(p).sum() for p in ('DRV_', 'VEH_', 'ACS_')})
auc.where(auc >= .5, 1 - auc).sort_values(ascending=False).head(20).round(3).to_frame('single-feature AUC')

(36871, 83) | new columns by prefix: {'DRV_': np.int64(19), 'VEH_': np.int64(15), 'ACS_': np.int64(18)}


,single-feature AUC
QUOTED_AMT,0.571
CAT_ZONE,0.569
CREDIT_SCORE,0.565
DRV_HIGH_EDU_SHARE,0.563
DRV_YOUNG_SHARE,0.559
DRV_DEPENDENT_SHARE,0.559
ACS_POP_DENSITY,0.557
ACS_COMMUTE_TRANSIT,0.555
ACS_COMMUTE_DROVE_ALONE,0.547
ACS_COMMUTE_MEAN_MIN,0.545
